# FontDiffusion Cache Generator (Google Colab T4)

**Runtime > Change runtime type > T4 GPU**

Gen anh viet tay co cho tier 3 candidates. Upload `tier3_chars.txt` + `tier3_styles.zip` tu may local.

In [ ]:
# 1. Cai dependencies
!pip install -q torch torchvision diffusers accelerate safetensors einops info-nce-pytorch kornia datasets pygame huggingface-hub
print('Done install')

In [ ]:
# 2. Clone FontDiffusion source code
!git clone https://github.com/dzungphieuluuky/FontDiffusion.git
%cd FontDiffusion
!ls

In [ ]:
# 3. Download model checkpoint tu HuggingFace
from huggingface_hub import hf_hub_download

repo = 'dzungpham/font-architect'

fst_files = [
    'FST/checkpoint_step_1500/content_encoder.safetensors',
    'FST/checkpoint_step_1500/fst_module.safetensors',
    'FST/checkpoint_step_1500/fst_projection.safetensors',
    'FST/checkpoint_step_1500/mss_encoder.safetensors',
    'FST/checkpoint_step_1500/original_style_projection.safetensors',
    'FST/checkpoint_step_1500/style_encoder.safetensors',
    'FST/checkpoint_step_1500/unet.safetensors',
]
dro_files = [
    'DRO-20260227-19P2/checkpoint_step_6000/content_encoder.safetensors',
    'DRO-20260227-19P2/checkpoint_step_6000/fst_module.safetensors',
    'DRO-20260227-19P2/checkpoint_step_6000/fst_projection.safetensors',
    'DRO-20260227-19P2/checkpoint_step_6000/mss_encoder.safetensors',
    'DRO-20260227-19P2/checkpoint_step_6000/original_style_projection.safetensors',
    'DRO-20260227-19P2/checkpoint_step_6000/scr.safetensors',
    'DRO-20260227-19P2/checkpoint_step_6000/style_encoder.safetensors',
    'DRO-20260227-19P2/checkpoint_step_6000/unet.safetensors',
]

all_files = fst_files + dro_files
for i, f in enumerate(all_files):
    print(f'[{i+1}/{len(all_files)}] {f.split("/")[-1]}')
    hf_hub_download(repo_id=repo, filename=f, local_dir='ckpt')
print('Checkpoint downloaded!')

In [ ]:
# 4. Upload tier3_chars.txt va tier3_styles.zip tu may local
#
# Truoc khi upload, tren may local chay:
#   cd prepared/CacThanhTruyen2
#   zip -r tier3_styles.zip tier3_styles/
#
from google.colab import files
print('Upload 2 file: tier3_chars.txt va tier3_styles.zip')
uploaded = files.upload()

In [ ]:
# 5. Giai nen styles va doc chars
!unzip -qo tier3_styles.zip

from pathlib import Path

with open('tier3_chars.txt', 'r', encoding='utf-8') as f:
    chars = [line.strip() for line in f if line.strip()]

style_files = sorted(Path('tier3_styles').glob('*.png'))
print(f'Characters: {len(chars)}')
print(f'Style images: {len(style_files)}')
for s in style_files:
    print(f'  {s.name}')

In [ ]:
# 6. Fix API mismatch trong FontDiffusion code
with open('inference/sample_optimized.py', 'r') as f:
    content = f.read()
if 'num_inference_steps=getattr' in content:
    content = content.replace('num_inference_steps=getattr', 'num_inference_step=getattr')
    with open('inference/sample_optimized.py', 'w') as f:
        f.write(content)
    print('Fixed API mismatch')
else:
    print('Already fixed')

In [ ]:
# 7. Check GPU
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')
else:
    print('KHONG CO GPU! Vao Runtime > Change runtime type > T4 GPU')

In [ ]:
# 8. Load pipeline (1 lan duy nhat)
import time
from PIL import Image

from src.configs.fontdiffuser import get_parser
from inference.sample_optimized import (
    load_fontdiffuser_pipeline, FontManager,
    get_style_transform, get_content_transform,
)
from src.tools.utils import ttf2im

parser = get_parser()
args = parser.parse_args([])

# Config cho T4 GPU
args.ckpt_dir = 'ckpt/DRO-20260227-19P2/checkpoint_step_6000'
args.phase_1_ckpt_dir = 'ckpt/FST/checkpoint_step_1500'
args.fst_ckpt_path = args.ckpt_dir
args.ttf_path = 'fonts/NomNaTong-Regular.ttf'
args.device = 'cuda'
args.use_fst = True
args.fp16 = True       # T4 ho tro FP16 tot
args.batch_size = 16   # T4 15GB VRAM, batch 16 ok
args.style_image_size = (args.style_image_size, args.style_image_size)
args.content_image_size = (args.content_image_size, args.content_image_size)

print(f'Device: {args.device}, FP16: {args.fp16}, Batch: {args.batch_size}')

pipe = load_fontdiffuser_pipeline(args=args, use_fst=True)
font_manager = FontManager(args.ttf_path)
font_name = font_manager.get_font_names()[0]
font = font_manager.get_font(font_name)

# Pre-render content images (dung chung cho moi style)
content_transform = get_content_transform(args.content_image_size)
available = font_manager.get_available_chars_for_font(font_name, chars)

content_tensors = []
valid_chars = []
for c in available:
    try:
        img = ttf2im(font=font, char=c)
        if img is not None:
            content_tensors.append(content_transform(img))
            valid_chars.append(c)
    except:
        pass

print(f'Chars in font: {len(valid_chars)}/{len(chars)}')
print(f'Styles: {len(style_files)} pages')
print(f'Total images to generate: {len(valid_chars)} x {len(style_files)} = {len(valid_chars) * len(style_files)}')
print('Pipeline ready!')

In [ ]:
# 9. Generate cho moi style image
BATCH_SIZE = args.batch_size
CACHE_DIR = Path('fd_cache')
style_transform = get_style_transform(args.style_image_size)
dtype = torch.float16

grand_start = time.time()
grand_total = 0
grand_target = len(valid_chars) * len(style_files)

for style_idx, style_path in enumerate(style_files):
    style_name = style_path.stem  # page_0012, page_0014, ...
    out_dir = CACHE_DIR / style_name
    out_dir.mkdir(parents=True, exist_ok=True)

    # Skip da cached
    to_gen_idx = []
    for i, c in enumerate(valid_chars):
        if not (out_dir / f'U+{ord(c):04X}.png').exists():
            to_gen_idx.append(i)

    cached = len(valid_chars) - len(to_gen_idx)
    if not to_gen_idx:
        print(f'[Style {style_idx+1}/{len(style_files)}] {style_name}: {cached} cached, skip')
        grand_total += cached
        continue

    print(f'\n{"="*60}')
    print(f'[Style {style_idx+1}/{len(style_files)}] {style_name}')
    print(f'  To generate: {len(to_gen_idx)} ({cached} cached)')
    print(f'  Grand progress: {grand_total}/{grand_target}')
    print(f'{"="*60}')

    style_image = Image.open(str(style_path)).convert('RGB')
    style_tensor = style_transform(style_image)

    generated = 0
    start = time.time()
    n_batches = (len(to_gen_idx) + BATCH_SIZE - 1) // BATCH_SIZE

    for b in range(0, len(to_gen_idx), BATCH_SIZE):
        batch_idx = to_gen_idx[b:b+BATCH_SIZE]
        batch_chars = [valid_chars[i] for i in batch_idx]
        bc = torch.stack([content_tensors[i] for i in batch_idx]).to('cuda', dtype=dtype)
        bs = style_tensor[None,:].repeat(len(batch_chars),1,1,1).to('cuda', dtype=dtype)

        with torch.inference_mode():
            images = pipe.generate(
                content_images=bc, style_images=bs, batch_size=len(bc),
                order=args.order,
                num_inference_step=args.num_inference_steps,
                content_encoder_downsample_size=args.content_encoder_downsample_size,
                t_start=args.t_start, t_end=args.t_end,
                dm_size=args.content_image_size,
                algorithm_type=args.algorithm_type,
                skip_type=args.skip_type,
                method=args.method,
                correcting_x0_fn=args.correcting_x0_fn,
            )

        for c, img in zip(batch_chars, images):
            img.save(str(out_dir / f'U+{ord(c):04X}.png'))

        generated += len(images)
        grand_total += len(images)
        elapsed = time.time() - start
        rate = elapsed / generated
        remaining_style = rate * (len(to_gen_idx) - generated)
        remaining_total = rate * (grand_target - grand_total)
        batch_num = b // BATCH_SIZE + 1
        print(f'  [{batch_num}/{n_batches}] {generated}/{len(to_gen_idx)} '
              f'({rate:.2f}s/img) '
              f'style ~{remaining_style/60:.0f}min | '
              f'total ~{remaining_total/60:.0f}min left')

        del bc, bs, images
        torch.cuda.empty_cache()

grand_elapsed = time.time() - grand_start
print(f'\n{"="*60}')
print(f'DONE: {grand_total} images, {len(style_files)} styles, {grand_elapsed/60:.1f} min')
print(f'{"="*60}')

In [ ]:
# 10. Zip va download
import shutil, os

shutil.make_archive('fd_cache', 'zip', '.', 'fd_cache')
n_files = len(list(CACHE_DIR.rglob('*.png')))
zip_size = os.path.getsize('fd_cache.zip') / 1024 / 1024
print(f'fd_cache.zip: {n_files} images, {zip_size:.1f} MB')

files.download('fd_cache.zip')

## Sau khi download

Tren may local:
```bash
# Giai nen vao dung cho
unzip fd_cache.zip -d prepared/CacThanhTruyen2/fd_cache/

# Chay pipeline — FontDiffusion cache hit instant
./run_pipeline.sh --book CacThanhTruyen2
```

Cache structure:
```
prepared/CacThanhTruyen2/fd_cache/
  page_0012/U+7D93.png   <- chu kinh, style trang 12
  page_0014/U+7D93.png   <- chu kinh, style trang 14
  ...
```

Tier 3 se so crop tu trang 12 voi anh gen style trang 12 (cung phong cach viet).